# Step 20.5 — Resume Verification

## 恢复后“能运行”还不够：与哪一份参考、按什么规则比较？

20.4 建立恢复流程，本节只讨论 **small deterministic test（固定小输入核对）** 与 **mismatch detection（不匹配检测）**。关键不是再训练一次，而是保留同一保存边界的参考对象，分清结构、近似数值与精确计数三类检查。

先在仓库根目录同步：

```bash
git pull
```

再安全复制成独立时间戳 work 副本：

```bash
mkdir -p notebooks/work && cp -n notebooks/91_step20_5_resume_verification.ipynb "notebooks/work/91_step20_5_resume_verification_$(date +%Y%m%d_%H%M%S)_work.ipynb"
```

本母版代码全未执行、无输出；本次不读取语料、不加载 checkpoint、不运行模型、测试、训练或测量。以下数字均为纸面推导。Python 3.12、MLX v0.32.2 保持；旧完整模型放在可选复用附录，实际调用前才须具备全部定义。

## 1. 先建立可信 reference：固定输入不能弥补参考对象错了

**Reference（参考）**必须代表保存时刻的状态；**restored（恢复对象）**是根据保存产物重新构造并加载的独立对象。设第 7 次更新后保存了模型 $\theta_7$ 与 Adam 状态 $S_7$：

| 时刻 | 应保留的参考 | 恢复对象 | 本次应比较什么？ |
|---|---|---|---|
| 保存边界 | 不再变化的 $\theta_7,S_7$ | 尚未建立 | 记录配置、映射、进度与文件来源 |
| 恢复完成、尚未新 update | 仍为 $\theta_7,S_7$ | $\widehat\theta_7,\widehat S_7$ | 同一边界的参数、状态、小输入输出与计数 |
| 若参考继续训练到第 8 步 | 已变成 $\theta_8,S_8$ | 仍为第 7 步 | 此时比较失败不能直接归因于保存/加载错误 |

反例：`reference_model = model` 只是给同一个对象增加名字，不是保存快照；`reference_model = restored_model` 更是在拿对象和自己比较。即使所有检查都通过，也没证明保存前后相同。`model.parameters()` 返回的树也不能未经确认就当作永不改变的独立快照。

未来操作应在保存边界保留独立、稳定的参考对象/快照，并记录它所属的更新数；在验证完成前，不再更新参考或恢复对象。没有这样的参考，就只能报告“完成了文件/结构检查”，不能补造“保存前后数值一致”。从同一个 checkpoint 再加载两份对象互比，只能支持两次加载的一致性，不能代替保存前参考。

特别注意：本节的 `compare_logits(reference_model, restored_model, X)` **会在调用时重新对两个模型做 forward**，不接收一份先前缓存的 logits。即使之前已有 `reference_logits` 变量，它也不会自动使用那份快照。若只剩保存的 logits 而没有参考模型，当前接口不足以完成这项比较，需另行明确验证方式；本节不新增实现。

## 2. 小配置与固定输入：缩小检查范围，不替换真实 checkpoint

`VERIFY_CONFIG` 是独立小验证候选：V=11、C=16、H=4、一个 Block、位置容量 8。以后只有按这套结构建立/保存的验证产物才能按它恢复；真实课程 checkpoint 应用自己的实际配置，不能强行加载进小模型。

下面只声明配置，不创建模型或 optimizer。`VERIFY_ADAM_CONFIG` 同样不是从文件自动发现的原运行超参数；实际配对还要按 20.3/20.4 的记录核对。

In [ ]:
import json
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten

# Tiny deterministic verification configuration; not a training configuration.
VERIFY_CONFIG = {
    "vocab_size": 11,
    "dims": 16,
    "num_heads": 4,
    "num_blocks": 1,
    "max_seq_len": 8,
}
VERIFY_ADAM_CONFIG = {
    "learning_rate": 3e-4,
    "betas": [0.9, 0.999],
    "eps": 1e-8,
    "bias_correction": True,
}

### Architecture 比较是第一道门，不是完整配置认证

`assert_architecture_compatible` 比较五个字段的值，将不一致项收集为 `字段: (保存值,当前值)`。若存在差异就抛 ValueError；若字段一致，再检查 dims 能否被 num_heads 整除。

例如保存 dims=16、当前 dims=32，即使都能被 4 整除，仍会因 width 不同被拒绝。保存 V=11、当前 V=12 也同样不兼容。先做这种核对，避免把文件加载异常当作第一条提示。

此函数假设输入为具备有效字段的配置字典，不完整检查所有类型/正数，也不检查 tokenizer、Adam 超参数或模型实际属性。两个都缺失的字段不能靠 `.get` 相等证明有效，应先有完整的配置记录。`require` 是保留的简单断言辅助函数；下面的具体比较函数使用自己的异常检查，不是全部通过它执行。

In [ ]:
def require(condition, message):
    if not condition:
        raise AssertionError(message)

def assert_architecture_compatible(saved_config, current_config):
    keys = ("vocab_size", "dims", "num_heads", "num_blocks", "max_seq_len")
    mismatches = {
        key: (saved_config.get(key), current_config.get(key))
        for key in keys if saved_config.get(key) != current_config.get(key)
    }
    if mismatches:
        raise ValueError(f"Architecture/config mismatch: {mismatches}")
    if current_config["dims"] % current_config["num_heads"] != 0:
        raise ValueError("dims must be divisible by num_heads.")
    return True

### Negative case：确认错误确实会被拒绝

`expect_architecture_mismatch` 调用配置检查：捕获预期的 ValueError 并返回说明；若没有出现异常，反而抛 AssertionError。下面注释里只改变 width 或 V，因此检验的是这两种不匹配，而不是“所有坏配置都能识别”。当前未执行任何负向测试。

In [ ]:
def expect_architecture_mismatch(saved_config, candidate_config):
    try:
        assert_architecture_compatible(saved_config, candidate_config)
    except ValueError as exc:
        return str(exc)
    raise AssertionError("Expected an architecture/config mismatch, but none was detected.")

# Illustrative negative cases; calls are examples for the future learner, not run here.
# wrong_width = dict(VERIFY_CONFIG, dims=32)
# expect_architecture_mismatch(VERIFY_CONFIG, wrong_width)
# wrong_vocab = dict(VERIFY_CONFIG, vocab_size=12)
# expect_architecture_mismatch(VERIFY_CONFIG, wrong_vocab)

### 固定 X 的 shape 走一遍

`make_verification_input` 定义两行、每行五个 token ID，shape 为 `(2,5)`；数值 1–5 都在本小配置合法范围 0–10 内，长度 5 不超过容量 8。两行不是随机采样，也不依赖 tokenizer 或 corpus。

| 阶段 | Shape | 本节观察含义 |
|---|---|---|
| token IDs X | `(2,5)` | 固定的两个小前缀 |
| token/position features | `(2,5,16)` | 小模型宽度为 16 |
| logits | `(2,5,11)` | 每个位置的 11 个 raw scores |

`make_verification_model` 只调用已有 MiniGPT 构造器，不加载权重。其后注释 setup 仅展示**初始化参考模型**与固定输入，不创建恢复对象，也不保存训练后的参考。重新设相同 seed 不能代替恢复真实已训练参数。

In [ ]:
def make_verification_input():
    # Fixed IDs, no tokenizer or corpus access.
    return mx.array(
        [[1, 2, 3, 4, 5],
         [5, 4, 3, 2, 1]],
        dtype=mx.int32,
    )

def make_verification_model(config):
    model = MiniGPT(
        vocab_size=config["vocab_size"],
        dims=config["dims"],
        num_heads=config["num_heads"],
        num_blocks=config["num_blocks"],
        max_seq_len=config["max_seq_len"],
    )
    return model

# Future work-copy setup only; not executed in this published master.
# mx.random.seed(2026)
# reference_model = make_verification_model(VERIFY_CONFIG)
# X_verify = make_verification_input()
# reference_logits = reference_model(X_verify)
# mx.eval(reference_model.parameters(), reference_logits)

## 3. Tree comparison：先结构和 dtype，再谈数值接近

`tree_flatten` 将参数/状态树变成 `(path, leaf)` 列表；这里的 path 标识某一叶子，而不是磁盘文件路径。`compare_array_trees` 按以下顺序检查：

1. 路径列表及其顺序必须一致，否则直接拒绝；仅有相同叶子数量不够。
2. 非数组叶子按类型和值检查。
3. 对数组先要求 shape 与 dtype 完全一致。
4. 求值后，用 `mx.allclose` 比较数组数值，返回 True 或在首个失败处抛异常。

纸面上，参考叶子 shape `(2,3)`、恢复叶子 `(3,2)`，即使都是 6 个元素也拒绝；参考 float32、恢复 float16，即使显示值相似也先拒绝。`step` 的 `()` 是零维标量数组，与 `(1,)` 不同。

### allclose 的 atol 与 rtol 分别允许什么？

对每个有限标量，MLX 固定版本采用：

$$
|a-b|\leq \mathrm{atol}+\mathrm{rtol}\,|b|.
$$

所有位置都满足才认为接近。这里 $a$ 是 reference 值，$b$ 是 restored 值；相对项依赖第二个输入，不能把这种检查当成精确相等或天然完全对称的距离。

- **atol（绝对容差）**给接近 0 的值留下固定小误差额度。
- **rtol（相对容差）**按数值规模允许一定比例差异。
- 本课 wrapper 默认 `rtol=1e-5, atol=1e-6`；这是本节选择，atol **不是** MLX `allclose` 自身默认的 `1e-8`。

只作十进制纸面算例：取恢复值 $b=2$，允许误差为 $10^{-6}+10^{-5}\times2=0.000021$。参考值若是 2.00001，差 0.00001 小于阈值；若是 2.0001，差 0.0001 超出阈值。当 $b=0$ 时阈值仅剩 0.000001。这不是实际浮点运行结果，也不表示这组容差适合所有模型和 dtype；不能为了通过就随意放宽。

[MLX v0.32.2 allclose 定义](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)还说明它支持 broadcasting。但此树 helper 先检查 shape，故不会借广播把不同 shape 当成同一参数；同号无穷在 allclose 中可能被视为相等，因此“接近”也不是参数有限/健康的完整检查。

In [ ]:
def compare_array_trees(reference_tree, restored_tree, *, rtol=1e-5, atol=1e-6):
    reference = tree_flatten(reference_tree)
    restored = tree_flatten(restored_tree)
    if [str(path) for path, _ in reference] != [str(path) for path, _ in restored]:
        raise AssertionError("Tree paths/order differ.")
    for (path_a, a), (path_b, b) in zip(reference, restored):
        if not isinstance(a, mx.array) or not isinstance(b, mx.array):
            if type(a) is not type(b) or a != b:
                raise AssertionError(f"Non-array leaf differs at {path_a}.")
            continue
        if tuple(a.shape) != tuple(b.shape) or a.dtype != b.dtype:
            raise AssertionError(f"Shape/dtype differs at {path_a}.")
        mx.eval(a, b)
        if not bool(mx.allclose(a, b, rtol=rtol, atol=atol).item()):
            raise AssertionError(f"Array values differ at {path_a}.")
    return True

## 4. Logits comparison：同一输入，两个真正独立的同版本模型

`compare_logits` 对 reference/restored 分别 forward 同一个 X，求值后先检查输出 shape，再做 allclose。对于本小配置，预期是 `(2,5,11)`；它检查**两者 shape 相等**，没有在 helper 里另写固定 `(2,5,11)` 断言，也没有像树比较那样显式要求 logits dtype 一样。

正确顺序是先做参数树/配置等核对，再用固定 X 检查前向路径。两组 logits 在这 10 个位置接近，不表示所有输入都相同，更不证明语言质量。相同参数也仍需相容模型代码和执行条件；当前模型没有 dropout，未来模型若增加随机或模式相关行为，还要控制这些因素，不能只固定 X。

此函数成功时只返回 True，不返回两份 logits 或最大绝对误差。如果未来报告需要最大差值，须另行真实测量、记录；不能把纸面预期或这个布尔结果当作已经测得的误差。

In [ ]:
def compare_logits(reference_model, restored_model, X, *, rtol=1e-5, atol=1e-6):
    reference_logits = reference_model(X)
    restored_logits = restored_model(X)
    mx.eval(reference_logits, restored_logits)
    if reference_logits.shape != restored_logits.shape:
        raise AssertionError(
            f"Logit shape mismatch: {reference_logits.shape} vs {restored_logits.shape}"
        )
    if not bool(mx.allclose(reference_logits, restored_logits, rtol=rtol, atol=atol).item()):
        raise AssertionError("Same-input logits differ beyond tolerance.")
    return True

## 5. Optimizer tree 接近，不代表整数计数已经精确一致

`verify_optimizer_state` 把两个 optimizer 的 state 交给同一个 `compare_array_trees`。它比较 m/v、step、learning_rate 等实际叶子，但**没有按整数 dtype 改成精确相等**：所有 MLX 数组（包括整数 step）都走 allclose。

因此不能把该函数成功表述成“所有整数状态逐值严格相同”。即使某组很小计数的差 1 通常能被拒绝，相对容差也不能普遍替代计数精确核对。

独立的 `read_adam_step` 展平状态树，寻找路径字符串以 `step` 结尾的叶子，要求恰好一个且是 shape `()` 的 MLX 数组，再求值并转成 Python int。它不单独验证这个数组一定是整数 dtype；当前协议依赖正确的 Adam 状态与前面的 dtype 核对，不能把 `int(...)` 转换本身当作类型认证。

`verify_progress(optimizer, metadata)` 再将这个整数与真实非负整数 `completed_updates` 用 `!=` 比较，这一层不是容差比较。

In [ ]:
def verify_optimizer_state(reference_optimizer, restored_optimizer):
    return compare_array_trees(reference_optimizer.state, restored_optimizer.state)

### 精确核对覆盖的是哪一个 optimizer？

每调用一次 `verify_progress`，只检查传入的那一个 optimizer。如果未来只传 restored_optimizer，它只证明恢复计数与 metadata 相符，**没有自动检查 reference_optimizer 也是同一保存步**。

在本节的同边界验证方案里，学习者未来应显式对 reference 与 restored 分别使用现有进度检查、对照同一份可信 metadata，并记录两者实际 step。纸面上，两者都应为 7；若 reference 已到 8，不能仅凭 restored=7 就宣称参考/恢复已完全一致。这是未来操作要求，不是已经执行过的检查，也不新增函数或调用。

In [ ]:
def read_adam_step(optimizer):
    flattened = tree_flatten(optimizer.state)
    matches = [(path, value) for path, value in flattened if str(path).endswith("step")]
    if len(matches) != 1:
        raise ValueError(f"Expected one Adam step leaf, found {len(matches)}.")
    path, value = matches[0]
    if not isinstance(value, mx.array) or value.shape != ():
        raise TypeError(f"Adam step must be a zero-dimensional MLX array, got {type(value)} / {getattr(value, 'shape', None)}.")
    mx.eval(value)
    return int(value.item())

def verify_progress(optimizer, metadata):
    expected = metadata.get("training_progress", {}).get("completed_updates")
    if type(expected) is not int or expected < 0:
        raise ValueError("Metadata completed_updates must be a real non-negative integer.")
    actual = read_adam_step(optimizer)
    if actual != expected:
        raise AssertionError(f"Adam step {actual} != metadata completed_updates {expected}.")
    return True

## 6. 未来完整阅读/操作顺序与可报告结论

先读主线，再查必要的附录。未来实际调用前定义全部 helpers 和类，并准备真实配对产物；本节没有替你完成这些外部前置。

1. 在明确保存边界保留稳定、独立的 reference model/optimizer，记录其配置、映射、进度及来源。
2. 依据 20.3/20.4 核对架构、tokenizer、Adam 配置、artifact 文件与 hash，恢复另一组对象；此时**先不执行下一次 update**。
3. 对参数树做路径/shape/dtype 检查和明确容差的数值比较。
4. 用同一固定 X 比较 logits；使用相同条件和未被改写的参考模型。
5. 比较 optimizer state tree；另对 reference/restored 各自精确核对 completed_updates，不用通用 allclose 代替整数核对。
6. 用已列出的错误 width/V 例子验证异常路径。任何未执行项记录为 not run，而不是预填 PASS。

未来报告应写实际配置、环境、输入、容差、各项检查和异常；如果记录最大差值，必须真的另外测量。可表述为“所列检查在指定输入/环境/容差下通过”，不能升级成训练质量、所有输入等价、完整随机状态恢复或跨设备逐位复现。

自查（不附答案代码）：

- 参考对象从 step 7 继续训练到 8 后，恢复比较失败说明什么？
- 为什么必须在 allclose 前先核对树路径、shape 和 dtype？
- `verify_optimizer_state` 通过但没核对 reference 的 step，缺少哪项证据？
- 只有保存过的 reference_logits、没有 reference_model，能直接使用当前 compare_logits 吗？

## 可选复用附录 · 旧完整 MiniGPT，不是本节新验证逻辑

以下保留 [15.4 完整模型](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/4f7ea87fa53b760c94049bf3e545b5b6b3d01a3c/notebooks/65_step15_4_implement_minigpt.ipynb)定义。主线阅读可以先跳过；未来调用 make_verification_model 之前必须完成这些定义。没有新模型算法、下载预训练模型、自动实例化或 forward。

### A. 原 Causal Multi-Head Attention

输入/输出 `(B,T,C)`，中间分头为 `(B,H,T,D)`，mask 仍在 forward 内创建；不需为本节重新逐行学习旧注意力实现。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)

### B. 原 MLP 与 Block

保留两层 MLP、GELU、Pre-Norm RMSNorm 残差结构，不改变参数命名与运算。

In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.

### C. 原完整 MiniGPT 汇总

构造器接收配置；forward 从 token IDs 返回 `(B,T,V)` raw logits。类定义与初始化都不是“已恢复训练权重”的证明，真实恢复仍走 20.4。

In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.

## 衔接与本次材料状态

- [20.2 optimizer 状态协议](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/4f7ea87fa53b760c94049bf3e545b5b6b3d01a3c/notebooks/88_step20_2_save_optimizer_state.ipynb)：树结构和 dtype 的来源。
- [20.3 metadata](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/4f7ea87fa53b760c94049bf3e545b5b6b3d01a3c/notebooks/89_step20_3_save_configuration_metadata.ipynb)：映射、hash、进度各自的范围。
- [20.4 恢复顺序](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/4f7ea87fa53b760c94049bf3e545b5b6b3d01a3c/notebooks/90_step20_4_resume_training.ipynb)：恢复后先验证，不急于下一次更新。

本次只改 20.5 讲解与组织，所有原函数/类/配置语义保留，未增加验证算法或可执行调用。历史修正计数不归零；本次等待独立静态验收，未把旧验收当作新结果。所有代码未执行、无输出；没有运行 Notebook、MLX、训练、测试、测量、语料或 checkpoint 操作。本节结束即停，不推进 Step 21。